In [2]:
import requests
import openpyxl
from openpyxl.styles import Alignment, Font, PatternFill
from openpyxl.drawing.image import Image as ExcelImage
from io import BytesIO

In [11]:
BASE_URL = "https://apis.data.go.kr/B551011/GoCamping/basedList?numOfRows=50&pageNo=1&MobileOS=ECT&MobileApp=asd&serviceKey=42xp7F1yttq7hugFYaNHPuEZ%2BFi90xA2R6m0NiZLDxWHpfG%2FpWwpDjgaRTGrOJLzkfYvJH%2FSIbCr1fXpjoeQPw%3D%3D&_type=json"
API_KEY = "42xp7F1yttq7hugFYaNHPuEZ+Fi90xA2R6m0NiZLDxWHpfG/pWwpDjgaRTGrOJLzkfYvJH/SIbCr1fXpjoeQPw=="
#API_KEY = "42xp7F1yttq7hugFYaNHPuEZ%2BFi90xA2R6m0NiZLDxWHpfG%2FpWwpDjgaRTGrOJLzkfYvJH%2FSIbCr1fXpjoeQPw%3D%3D"


params = {
    "numOfRows": 1000,
    "pageNo": 1,
    "MobileOS": "ETC",
    "MobileApp": "asd",
    "serviceKey": API_KEY,
    "_type": "json"
}

res = requests.get(BASE_URL, params=params)
data = res.json()

wb = openpyxl.Workbook()
ws = wb.active
ws.title = "캠핑장정보"

headers = ["아이디", "캠핑장명", "이미지", "지역", "주소", "홈페이지"]
ws.append(headers)

def px_to_row_height(px):
    return px * 0.75

def px_to_row_width(px):    
    return (px-14) / 7

# 내가 삽입하고자 하는 이미지 사이즈 (unit = pixel)
# excel의 크기 단위  = width : 글자 갯수, height = pt
IMG_H_PX = 75
IMG_W_PX = 100

ROW_H_PT = px_to_row_height(IMG_H_PX)
COL_W_PT = px_to_row_width(IMG_W_PX)


cell_align = Alignment(
    horizontal="left",
    vertical="center",
    wrap_text=True
)

header_fill = PatternFill(
    "solid",
    fgColor = "f2f2f2"
)

header_font = Font(bold=True)

# 첫번째 행 = 필드값이 들어가있는 공간
for col in range(1, len(headers) + 1) : 
    c = ws.cell(row=1, column=col)
    c.alignment = Alignment(
        horizontal="center",
        vertical="center"
    )
    c.fill = header_fill
    c.font = header_font

items = data["response"]["body"]["items"]["item"]
row = 2

for item in items:
    contentId = item.get("contentId", "")
    facltNm = item.get("facltNm", "")
    firstImageUrl = item.get("firstImageUrl", "")
    doNm = item.get("doNm", "")
    addr1 = item.get("addr1", "")
    homepage = item.get("homepage", "")
    
    ws.append([contentId, facltNm, "", doNm, addr1, homepage])
    
    ws.row_dimensions[row].height = ROW_H_PT
    
    for col in range(1, 7):
        ws.cell(row=row, column=col).alignment = cell_align

    if firstImageUrl : 
        try :
            img_data = requests.get(firstImageUrl).content # awropwgen2324982305489
            img_file = BytesIO(img_data)
            img = ExcelImage(img_file)
            img.width = IMG_W_PX
            img.height = IMG_H_PX
            ws.add_image(img, f"C{row}")

        except Exception as e :
            print(f"이미지 다운로드 실패 : {firstImageUrl} -> {e}")
    
    row += 1

ws.column_dimensions["A"].width = 12
ws.column_dimensions["B"].width = 30
ws.column_dimensions["C"].width = COL_W_PT
ws.column_dimensions["D"].width = 15
ws.column_dimensions["E"].width = 30
ws.column_dimensions["F"].width = 50

file_name = "camping_info.xlsx"

ws.freeze_panes = "A2"

wb.save(file_name)
print(f"✅ {file_name} 파일 저장 완료")   


✅ camping_info.xlsx 파일 저장 완료
